In [ ]:
from pathlib import Path
import sys

import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split

ROOT = Path.cwd()
SRC_DIR = ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from song_cleaning.joint_key_mode_geometric import (
    build_joint_coordinates,
    build_joint_distance_matrix,
    build_soft_targets,
    compute_joint_metrics,
    decode_joint_class,
    encode_joint_class,
    make_geometric_softprob_objective,
    softmax_from_margin,
)

DATA_PATH = ROOT / "liveness_final.parquet"
FEATURE_POOL = [
    "valence",
    "loudness",
    "danceability",
    "energy",
    "tempo",
    "acousticness",
    "instrumentalness",
    "liveness",
    "speechiness",
    "valence_missing",
    "loudness_missing",
    "danceability_missing",
    "energy_missing",
    "tempo_missing",
    "acousticness_missing",
    "instrumentalness_missing",
    "liveness_missing",
    "speechiness_missing",
    "key_missing",
    "mode_missing",
]
BASE_PARAMS = {
    "num_class": 24,
    "max_depth": 6,
    "eta": 0.03,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 1.0,
    "alpha": 0.0,
    "lambda": 1.0,
    "gamma": 0.0,
    "tree_method": "hist",
    "seed": 42,
    "disable_default_eval_metric": 1,
}
NUM_BOOST_ROUND = 600
ALPHA = 0.2
BETA = 1.0


In [ ]:
columns_to_load = sorted(set(FEATURE_POOL + ["key", "mode", "track_id", "artist_name", "track_name"]))
df = pd.read_parquet(DATA_PATH, columns=columns_to_load)
float64_columns = df.select_dtypes(include=["float64"]).columns
df[float64_columns] = df[float64_columns].astype("float32")
df[["key", "mode"]].isna().sum()


In [ ]:
observed_df = df[df["key"].notna() & df["mode"].notna()].copy()
missing_df = df[df["key"].isna() | df["mode"].isna()].copy()

X_observed = observed_df[FEATURE_POOL]
y_joint = encode_joint_class(observed_df["key"], observed_df["mode"])

X_train, X_test, y_train, y_test = train_test_split(
    X_observed,
    y_joint,
    test_size=0.2,
    random_state=42,
    stratify=y_joint,
)

dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)
len(X_train), len(X_test), len(missing_df)


In [ ]:
coordinates = build_joint_coordinates()
distance_matrix = build_joint_distance_matrix(coordinates)
soft_targets = build_soft_targets(distance_matrix=distance_matrix, alpha=ALPHA, beta=BETA)
custom_objective = make_geometric_softprob_objective(soft_targets)

baseline_params = dict(BASE_PARAMS)
baseline_params["objective"] = "multi:softprob"
baseline_model = xgb.train(baseline_params, dtrain, num_boost_round=NUM_BOOST_ROUND)

geometric_params = dict(BASE_PARAMS)
geometric_model = xgb.train(geometric_params, dtrain, num_boost_round=NUM_BOOST_ROUND, obj=custom_objective)


In [ ]:
baseline_margin = baseline_model.predict(dtest, output_margin=True)
baseline_probabilities = softmax_from_margin(baseline_margin)
baseline_joint_predictions = baseline_probabilities.argmax(axis=1)
baseline_decoded = decode_joint_class(baseline_joint_predictions)
baseline_true = decode_joint_class(y_test)

geometric_margin = geometric_model.predict(dtest, output_margin=True)
geometric_probabilities = softmax_from_margin(geometric_margin)
geometric_joint_predictions = geometric_probabilities.argmax(axis=1)
geometric_decoded = decode_joint_class(geometric_joint_predictions)

comparison = pd.DataFrame([
    {
        "model": "baseline_joint_classifier",
        **compute_joint_metrics(
            true_keys=baseline_true["predicted_key"],
            true_modes=baseline_true["predicted_mode"],
            predicted_keys=baseline_decoded["predicted_key"],
            predicted_modes=baseline_decoded["predicted_mode"],
            coordinates=coordinates,
        ),
    },
    {
        "model": "geometric_joint_classifier",
        **compute_joint_metrics(
            true_keys=baseline_true["predicted_key"],
            true_modes=baseline_true["predicted_mode"],
            predicted_keys=geometric_decoded["predicted_key"],
            predicted_modes=geometric_decoded["predicted_mode"],
            coordinates=coordinates,
        ),
    },
])
comparison


In [ ]:
dmissing = xgb.DMatrix(missing_df[FEATURE_POOL])
missing_margin = geometric_model.predict(dmissing, output_margin=True)
missing_probabilities = softmax_from_margin(missing_margin)
missing_joint_predictions = missing_probabilities.argmax(axis=1)
missing_decoded = decode_joint_class(missing_joint_predictions)

imputed_rows = missing_df[["track_id", "artist_name", "track_name"]].copy()
imputed_rows["key"] = missing_decoded["predicted_key"].to_numpy()
imputed_rows["mode"] = missing_decoded["predicted_mode"].to_numpy()
imputed_rows.head()
